# Task 4 — Build the Logical and Physical Data Model

Design a normalized structure from `staging_transactions` and `staging_line_items` (built in Task 3): a transaction entity, a line-item entity, and reference entities for customer, product, and store. Create the schema (from `sql/schema.sql`), load it from the staging tables, and verify referential integrity end to end.

**Design decisions** (full reasoning in `PROJECT_DOCUMENTATION.md`):

- Natural keys used where the data supports one: `customer_id`, `product_sku`, `store_id`. `invoice_number` is never a key -- Task 2 proved it isn't unique.
- `transaction_id` / `line_id` reuse the surrogate ids already produced in Task 3, so every row stays traceable back to reconstruction.
- Walk-ins (blank `customer_ref`) get no `dim_customer` row -- `fact_transaction.customer_id` is `NULL`, not a placeholder.
- `ST99` (in the export, not in `store_lookup.csv` -- Task 1) gets an explicit `dim_store` row with a placeholder name, instead of an orphaned FK or silently dropped transactions.
- `product_sku` / `qty` / `unit_price` on `fact_transaction_line` are nullable, because a handful of source rows (Task 1) have no resolvable value -- forcing a placeholder would misrepresent the data.

In [ ]:
import re
import sqlite3
import pandas as pd

conn = sqlite3.connect("../output/practice.db")

staging_lines = pd.read_sql("SELECT * FROM staging_line_items", conn)
staging_txn = pd.read_sql("SELECT * FROM staging_transactions", conn)

# check: should match Task 3's final numbers -- 131 lines, 63 transactions
print("staging_line_items rows:", len(staging_lines))
print("staging_transactions rows:", len(staging_txn))

## Create the schema

Run the actual T-SQL DDL file. SQLite's type affinity rules accept `NVARCHAR`, `DATETIME2`, `DECIMAL`, and `BIT` as written -- no SQLite-specific rewrite needed to prove the schema out here.

In [ ]:
with open("../sql/schema.sql") as f:
    schema_sql = f.read()

# drop tables first so this cell is safe to re-run
for tbl in ["fact_transaction_line", "fact_transaction", "dim_customer", "dim_product", "dim_store"]:
    conn.execute(f"DROP TABLE IF EXISTS {tbl}")

conn.executescript(schema_sql)
conn.commit()

# check: all 5 tables should now exist
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(tables)

## Build `dim_customer`

Use `customer_ref` as the key -- it's reliable even though the raw name/email spellings vary (Task 1 finding). For each customer, pick the most frequent non-blank raw value as the canonical name/email rather than trying to fuzzy-match spellings.

In [ ]:
def pick_canonical(series):
    """Most frequent non-blank value in a group -- used to pick one
    canonical spelling out of several raw variants for the same entity."""
    non_blank = series[series.notna() & (series != "")]
    if non_blank.empty:
        return None
    return non_blank.value_counts().idxmax()

# exclude walk-ins (blank customer_ref) -- they don't get a dim_customer row
cust_rows = staging_lines[staging_lines["customer_ref"].notna() & (staging_lines["customer_ref"] != "")]

dim_customer = (
    cust_rows.groupby("customer_ref")
    .agg(
        customer_name=("customer_name_raw", pick_canonical),
        customer_email=("customer_email_raw", pick_canonical),
    )
    .reset_index()
    .rename(columns={"customer_ref": "customer_id"})
)

# check: one row per distinct customer_id, canonical spellings picked
print("dim_customer rows:", len(dim_customer))
dim_customer

## Build `dim_product`

Same approach: `product_sku` is the reliable key, canonical name picked by frequency, category case-normalized (Task 1: 8 raw values collapse to 4 real categories).

In [ ]:
def title_case_category(series):
    non_blank = series[series.notna() & (series != "")]
    if non_blank.empty:
        return None
    return non_blank.str.lower().str.title().value_counts().idxmax()

prod_rows = staging_lines[staging_lines["product_sku"].notna() & (staging_lines["product_sku"] != "")]

dim_product = (
    prod_rows.groupby("product_sku")
    .agg(
        product_name=("product_name_raw", pick_canonical),
        category=("category_raw", title_case_category),
    )
    .reset_index()
)

# check: distinct SKU count and confirm category values look normalized now
print("dim_product rows:", len(dim_product))
print("distinct categories after normalization:", dim_product["category"].nunique())
dim_product

## Build `dim_store`

Start from `store_lookup.csv`, then add an explicit placeholder row for any store id used in the export but missing from the lookup table (Task 1: `ST99`) -- rather than an orphaned foreign key or silently dropped transactions.

In [ ]:
stores = pd.read_csv("../data/store_lookup.csv")

used_stores = set(staging_lines["store_id"].unique())
unmapped = used_stores - set(stores["store_id"])

placeholder_rows = pd.DataFrame({
    "store_id": list(unmapped),
    "store_name": ["UNKNOWN / UNMAPPED"] * len(unmapped),
})

dim_store = pd.concat([stores, placeholder_rows], ignore_index=True)

# check: every store_id used in the export should now exist in dim_store
print("dim_store rows:", len(dim_store))
print("still missing after placeholder step (should be empty):", used_stores - set(dim_store["store_id"]))
dim_store

## Build `fact_transaction` and `fact_transaction_line`

In [ ]:
fact_transaction = staging_txn.rename(columns={
    "reconstructed_txn_id": "transaction_id",
    "customer_ref": "customer_id",
    "invoice_numbers_seen": "invoice_number",
})[["transaction_id", "store_id", "register_id", "customer_id", "start_ts", "end_ts",
     "invoice_number", "needs_manual_review"]].copy()

# walk-ins: blank customer_ref becomes NULL, not an orphan FK to a
# dim_customer row that doesn't exist
fact_transaction["customer_id"] = fact_transaction["customer_id"].replace("", None)

# check: row count should match staging_transactions exactly (1:1 rename, no filtering)
print("fact_transaction rows:", len(fact_transaction))
fact_transaction.head()

In [ ]:
def parse_price(raw):
    """unit_price_raw is inconsistently formatted ($, spaces, no separator --
    Task 1 finding). Strip everything but digits and the decimal point."""
    if pd.isna(raw):
        return None
    cleaned = re.sub(r"[^\d.]", "", str(raw))
    return float(cleaned) if cleaned else None

fact_transaction_line = staging_lines.rename(columns={
    "row_id": "line_id",
    "reconstructed_txn_id": "transaction_id",
}).copy()

fact_transaction_line["unit_price"] = fact_transaction_line["unit_price_raw"].apply(parse_price)
fact_transaction_line["product_sku"] = fact_transaction_line["product_sku"].replace("", None)
fact_transaction_line["qty"] = pd.to_numeric(fact_transaction_line["qty"], errors="coerce")

fact_transaction_line = fact_transaction_line[[
    "line_id", "transaction_id", "product_sku", "qty", "unit_price",
    "discount_code", "payment_method", "line_note",
]]

# check: row count should match staging_line_items exactly, and unit_price
# should now be numeric with only the originally-missing rows still null
print("fact_transaction_line rows:", len(fact_transaction_line))
print("null unit_price after parsing:", fact_transaction_line["unit_price"].isna().sum())
fact_transaction_line.head()

## Load the tables

In [ ]:
dim_customer.to_sql("dim_customer", conn, if_exists="append", index=False)
dim_product.to_sql("dim_product", conn, if_exists="append", index=False)
dim_store.to_sql("dim_store", conn, if_exists="append", index=False)
fact_transaction.to_sql("fact_transaction", conn, if_exists="append", index=False)
fact_transaction_line.to_sql("fact_transaction_line", conn, if_exists="append", index=False)
conn.commit()

# check: row counts in the database should match what was built in memory above
for tbl in ["dim_customer", "dim_product", "dim_store", "fact_transaction", "fact_transaction_line"]:
    n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {tbl}", conn)["n"].iloc[0]
    print(f"{tbl}: {n} rows")

## Verify referential integrity

SQLite doesn't enforce foreign keys by default -- run explicit anti-join checks instead. Each of these should return an empty result; anything else is an orphaned foreign key that slipped through the ETL.

In [ ]:
checks = {
    "fact_transaction.store_id not in dim_store": """
        SELECT ft.transaction_id, ft.store_id FROM fact_transaction ft
        LEFT JOIN dim_store ds ON ft.store_id = ds.store_id
        WHERE ds.store_id IS NULL
    """,
    "fact_transaction.customer_id not in dim_customer (excluding NULL/walk-in)": """
        SELECT ft.transaction_id, ft.customer_id FROM fact_transaction ft
        LEFT JOIN dim_customer dc ON ft.customer_id = dc.customer_id
        WHERE ft.customer_id IS NOT NULL AND dc.customer_id IS NULL
    """,
    "fact_transaction_line.transaction_id not in fact_transaction": """
        SELECT ftl.line_id, ftl.transaction_id FROM fact_transaction_line ftl
        LEFT JOIN fact_transaction ft ON ftl.transaction_id = ft.transaction_id
        WHERE ft.transaction_id IS NULL
    """,
    "fact_transaction_line.product_sku not in dim_product (excluding NULL)": """
        SELECT ftl.line_id, ftl.product_sku FROM fact_transaction_line ftl
        LEFT JOIN dim_product dp ON ftl.product_sku = dp.product_sku
        WHERE ftl.product_sku IS NOT NULL AND dp.product_sku IS NULL
    """,
}

all_clean = True
for label, query in checks.items():
    result = pd.read_sql(query, conn)
    status = "OK (0 orphans)" if result.empty else f"FAILED ({len(result)} orphans)"
    if not result.empty:
        all_clean = False
    print(f"{label}: {status}")

print("\nAll referential integrity checks passed" if all_clean else "\nSome checks failed -- see above")

In [ ]:
conn.close()